In [1]:
import json
import pandas as pd
with open('/root/autodl-tmp/train_moe_from_scratch/20000step_trainer_state.json' , 'r', encoding='utf-8') as f:
    data = json.load(f)
    log_history = data['log_history']
    steps = []
    losses = []
    for i in log_history[:-1]:
        step = i['step']
        loss = i['loss']
        steps.append(step)
        losses.append(loss)

import seaborn as sns
import plotly.express as px

# 创建一个DataFrame
df = pd.DataFrame({
    'Step': steps,
    'Loss': losses
})

# 使用Plotly绘制散点图
fig = px.line(df, x='Step', y='Loss', title='MoE预训练损失变化',
                 labels={'Step': 'step', 'Loss': 'loss'},
                 )  # 添加趋势线可选

fig.update_layout(width=600, height=400)
# 显示图表
fig.show()

FileNotFoundError: [Errno 2] No such file or directory: '/root/autodl-tmp/train_moe_from_scratch/20000step_trainer_state.json'

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

/root/miniconda3/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
x = torch.randn(2, 4, 10)

In [4]:
# 充当专家
gate = nn.Linear(10, 4)

In [5]:
logits = gate(x)
logits.shape

torch.Size([2, 4, 4])

In [6]:
logits

tensor([[[ 0.1292, -0.6850,  0.3223,  0.0278],
         [-0.4622,  1.4601, -0.6874,  0.0672],
         [ 0.7171, -0.7665, -1.1472,  0.6078],
         [ 0.3628, -0.5434,  0.7117, -0.4804]],

        [[ 0.5293, -0.6636, -0.2316,  0.0656],
         [ 0.4249,  0.4631, -0.3345, -0.0924],
         [-0.0396, -0.3459, -1.4463, -0.4624],
         [-0.1882, -0.2087, -0.0369, -0.4318]]], grad_fn=<ViewBackward0>)

In [7]:
logits_topk, indices = logits.topk(2, dim=-1)
# 专家对应的概率和专家索引
logits_topk, indices

(tensor([[[ 0.3223,  0.1292],
          [ 1.4601,  0.0672],
          [ 0.7171,  0.6078],
          [ 0.7117,  0.3628]],
 
         [[ 0.5293,  0.0656],
          [ 0.4631,  0.4249],
          [-0.0396, -0.3459],
          [-0.0369, -0.1882]]], grad_fn=<TopkBackward0>),
 tensor([[[2, 0],
          [1, 3],
          [0, 3],
          [2, 0]],
 
         [[0, 3],
          [1, 0],
          [0, 1],
          [2, 0]]]))

In [8]:
zeros = torch.full_like(logits, float("-inf"))

In [9]:
sparse_logits = zeros.scatter(dim=-1, index=indices, src=logits_topk)
sparse_logits

tensor([[[ 0.1292,    -inf,  0.3223,    -inf],
         [   -inf,  1.4601,    -inf,  0.0672],
         [ 0.7171,    -inf,    -inf,  0.6078],
         [ 0.3628,    -inf,  0.7117,    -inf]],

        [[ 0.5293,    -inf,    -inf,  0.0656],
         [ 0.4249,  0.4631,    -inf,    -inf],
         [-0.0396, -0.3459,    -inf,    -inf],
         [-0.1882,    -inf, -0.0369,    -inf]]], grad_fn=<ScatterBackward0>)

In [10]:
# 将 token 选出来的 当前专家 对应的概率值填入对应的位置
# 归一化概率和=1
sparse_logits = F.softmax(sparse_logits, dim=-1)
# 意味着门控分数总和=1
sparse_logits

tensor([[[0.4519, 0.0000, 0.5481, 0.0000],
         [0.0000, 0.8011, 0.0000, 0.1989],
         [0.5273, 0.0000, 0.0000, 0.4727],
         [0.4136, 0.0000, 0.5864, 0.0000]],

        [[0.6139, 0.0000, 0.0000, 0.3861],
         [0.4905, 0.5095, 0.0000, 0.0000],
         [0.5760, 0.4240, 0.0000, 0.0000],
         [0.4622, 0.0000, 0.5378, 0.0000]]], grad_fn=<SoftmaxBackward0>)

In [11]:
final_outputs = torch.zeros_like(x)
final_outputs

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])

In [12]:
x_flat = x.view(-1, x.shape[-1])
x_flat, x_flat.shape

(tensor([[ 0.1834,  1.6143,  1.7607,  0.6381, -0.3468, -0.6350, -0.5403,  0.0144,
           1.1105,  0.5296],
         [ 1.0395, -1.6056, -1.2683,  0.3832, -1.0391,  0.0030, -0.7328, -0.2442,
          -2.8269, -0.4809],
         [ 0.4630, -0.0068, -0.2945, -2.7322, -0.4390, -1.6763,  0.2790,  1.1397,
           0.5131,  1.0050],
         [-1.4467,  0.7495,  1.7946,  0.1569,  0.8959, -0.1317, -0.4175,  0.4323,
           0.4290,  0.6569],
         [-0.6025,  0.6453,  0.0401, -0.1544, -1.9554, -0.3627,  0.8234,  2.0516,
           1.0682,  0.1396],
         [-0.4733,  0.1306, -0.5962,  0.0861, -0.2973,  0.1112,  0.5081,  1.2022,
          -1.9536, -0.0268],
         [ 0.6716,  1.0917, -1.2727,  0.3485,  0.2954,  0.7162,  1.5316, -1.4829,
           0.4969, -1.9118],
         [-0.4748, -0.5912,  0.8968, -0.6472, -1.4625, -0.2705,  1.5368,  0.5264,
          -0.9612,  0.6288]]),
 torch.Size([8, 10]))

In [13]:
sparse_logits_flat = sparse_logits.view(-1, sparse_logits.shape[-1])
sparse_logits_flat, sparse_logits_flat.shape

(tensor([[0.4519, 0.0000, 0.5481, 0.0000],
         [0.0000, 0.8011, 0.0000, 0.1989],
         [0.5273, 0.0000, 0.0000, 0.4727],
         [0.4136, 0.0000, 0.5864, 0.0000],
         [0.6139, 0.0000, 0.0000, 0.3861],
         [0.4905, 0.5095, 0.0000, 0.0000],
         [0.5760, 0.4240, 0.0000, 0.0000],
         [0.4622, 0.0000, 0.5378, 0.0000]], grad_fn=<ViewBackward0>),
 torch.Size([8, 4]))

In [14]:
expert_mask = (indices == 0).any(-1)
expert_mask.shape

torch.Size([2, 4])

In [15]:
expert_mask_flat = expert_mask.view(-1)
# 筛选出为 True 的专家
expert_mask_flat

tensor([ True, False,  True,  True,  True,  True,  True,  True])

In [16]:
expert_input = x_flat[expert_mask_flat]
# 有3个 token 须要处理
expert_input, expert_input.shape

(tensor([[ 0.1834,  1.6143,  1.7607,  0.6381, -0.3468, -0.6350, -0.5403,  0.0144,
           1.1105,  0.5296],
         [ 0.4630, -0.0068, -0.2945, -2.7322, -0.4390, -1.6763,  0.2790,  1.1397,
           0.5131,  1.0050],
         [-1.4467,  0.7495,  1.7946,  0.1569,  0.8959, -0.1317, -0.4175,  0.4323,
           0.4290,  0.6569],
         [-0.6025,  0.6453,  0.0401, -0.1544, -1.9554, -0.3627,  0.8234,  2.0516,
           1.0682,  0.1396],
         [-0.4733,  0.1306, -0.5962,  0.0861, -0.2973,  0.1112,  0.5081,  1.2022,
          -1.9536, -0.0268],
         [ 0.6716,  1.0917, -1.2727,  0.3485,  0.2954,  0.7162,  1.5316, -1.4829,
           0.4969, -1.9118],
         [-0.4748, -0.5912,  0.8968, -0.6472, -1.4625, -0.2705,  1.5368,  0.5264,
          -0.9612,  0.6288]]),
 torch.Size([7, 10]))

In [17]:
expert = nn.Linear(10, 10)

In [18]:
export_output = expert(expert_input)
export_output, export_output.shape

(tensor([[-0.7158,  0.3556,  0.0421, -0.0134, -1.0575,  0.0548,  0.1251,  0.6233,
          -1.3625,  0.0044],
         [ 0.8947,  1.2271,  0.4885, -0.3600, -0.3957,  0.2807,  1.0594,  0.4059,
          -0.0320,  0.0659],
         [-0.5673,  0.2983,  0.1554, -0.4873, -0.8003, -0.4105, -0.1632,  0.8452,
          -1.2964, -0.0131],
         [-0.4695,  1.0859,  0.0548,  0.0532, -0.3754,  0.3660,  0.9622,  0.7828,
          -0.8091, -0.8472],
         [ 0.3388,  0.0333, -0.5930,  0.5385, -0.3302,  0.6379, -0.0992,  0.5724,
           0.1311, -0.4626],
         [ 0.2309, -0.7336, -0.3464, -0.1667,  0.2180,  0.9083,  0.7065, -0.3214,
           1.2075,  0.7676],
         [-0.2261,  1.4573,  0.4474,  0.3713, -0.2057,  0.5434,  0.6002,  0.4574,
          -0.0467,  0.2763]], grad_fn=<AddmmBackward0>),
 torch.Size([7, 10]))

In [19]:
# 门控分数
gate_scores = sparse_logits_flat[expert_mask_flat, 0].unsqueeze(1)
gate_scores

tensor([[0.4519],
        [0.5273],
        [0.4136],
        [0.6139],
        [0.4905],
        [0.5760],
        [0.4622]], grad_fn=<UnsqueezeBackward0>)

In [20]:
# 当前专家处理 token 的输出对最终输出的影响权重
weighted_output = export_output * gate_scores
weighted_output, weighted_output.shape

(tensor([[-0.3234,  0.1607,  0.0190, -0.0060, -0.4779,  0.0248,  0.0565,  0.2816,
          -0.6156,  0.0020],
         [ 0.4718,  0.6470,  0.2576, -0.1898, -0.2087,  0.1480,  0.5586,  0.2140,
          -0.0169,  0.0348],
         [-0.2346,  0.1234,  0.0643, -0.2016, -0.3310, -0.1698, -0.0675,  0.3496,
          -0.5362, -0.0054],
         [-0.2882,  0.6666,  0.0337,  0.0326, -0.2305,  0.2247,  0.5907,  0.4805,
          -0.4967, -0.5201],
         [ 0.1662,  0.0163, -0.2908,  0.2641, -0.1619,  0.3128, -0.0486,  0.2808,
           0.0643, -0.2269],
         [ 0.1330, -0.4225, -0.1995, -0.0960,  0.1255,  0.5231,  0.4069, -0.1851,
           0.6955,  0.4421],
         [-0.1045,  0.6736,  0.2068,  0.1716, -0.0951,  0.2512,  0.2774,  0.2114,
          -0.0216,  0.1277]], grad_fn=<MulBackward0>),
 torch.Size([7, 10]))

In [21]:
final_outputs = torch.zeros_like(x)
final_outputs

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])

In [22]:
# 将乘以权重之后的 专家输出 填充到对应位置（对应的当前专家，对应的 token）
final_outputs[expert_mask] += weighted_output
final_outputs

tensor([[[-0.3234,  0.1607,  0.0190, -0.0060, -0.4779,  0.0248,  0.0565,
           0.2816, -0.6156,  0.0020],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000],
         [ 0.4718,  0.6470,  0.2576, -0.1898, -0.2087,  0.1480,  0.5586,
           0.2140, -0.0169,  0.0348],
         [-0.2346,  0.1234,  0.0643, -0.2016, -0.3310, -0.1698, -0.0675,
           0.3496, -0.5362, -0.0054]],

        [[-0.2882,  0.6666,  0.0337,  0.0326, -0.2305,  0.2247,  0.5907,
           0.4805, -0.4967, -0.5201],
         [ 0.1662,  0.0163, -0.2908,  0.2641, -0.1619,  0.3128, -0.0486,
           0.2808,  0.0643, -0.2269],
         [ 0.1330, -0.4225, -0.1995, -0.0960,  0.1255,  0.5231,  0.4069,
          -0.1851,  0.6955,  0.4421],
         [-0.1045,  0.6736,  0.2068,  0.1716, -0.0951,  0.2512,  0.2774,
           0.2114, -0.0216,  0.1277]]], grad_fn=<IndexPutBackward0>)